In [ ]:
from pathlib import Path
import hashlib

# Path to the folder
base_dir = Path(r"s:\src\unknown-horizons\content\gui\icons\tabwidget\buildmenu")
# base_dir = Path(r"s:\src\unknown-horizons\content\gui\icons\buildmenu")

def file_hash(path, chunk_size=8192):
    """Compute MD5 hash for a file."""
    h = hashlib.md5()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

# Step 1: Collect hashes
hash_map = {}
for file_path in base_dir.rglob('*'):
    if file_path.is_file():
        h = file_hash(file_path)
        hash_map.setdefault(h, []).append(file_path)

# Step 2: Print duplicates
duplicates_found = False
for h, files in hash_map.items():
    if len(files) > 1:
        duplicates_found = True
        print(f"\nDuplicate group (hash={h}):")
        for f in files:
            print("  ", f)

if not duplicates_found:
    print("✅ No duplicate files found.")

In [ ]:
# copy files with names
from pathlib import Path
import shutil

building_ids = {
  "WAREHOUSE"        :  1,
  "STORAGE"          :  2,
  "RESIDENTIAL"      :  3,
  "MAIN_SQUARE"      :  4,
  "PAVILION"         :  5,
  "SIGNAL_FIRE"      :  6,
  "WEAVER"           :  7,
  "LUMBERJACK"       :  8,
  "HUNTER"           :  9,
  "SETTLER_RUIN"     : 10,
  "FISHER"           : 11,
  "BOAT_BUILDER"     : 12,
  "LOOKOUT"          : 13,
  "TRAIL"            : 15,
  "TREE"             : 17,
  "PASTURE"          : 18,
  "POTATO_FIELD"     : 19,
  "FARM"             : 20,
  "VILLAGE_SCHOOL"   : 21,
  "SUGARCANE_FIELD"  : 22,
  "CLAY_DEPOSIT"     : 23,
  "BRICKYARD"        : 24,
  "CLAY_PIT"         : 25,
  "DISTILLERY"       : 26,
  "MINE"             : 28,
  "SMELTERY"         : 29,
  "TOOLMAKER"        : 30,
  "CHARCOAL_BURNER"  : 31,
  "TAVERN"           : 32,
  "FISH_DEPOSIT"     : 33,
  "MOUNTAIN"         : 34,
  "SALT_PONDS"       : 35,
  "TOBACCO_FIELD"    : 36,
  "TOBACCONIST"      : 37,
  "CATTLE_RUN"       : 38,
  "PIGSTY"           : 39,
  "HERBARY"          : 40,
  "BUTCHERY"         : 41,
  "DOCTOR"           : 42,
  "GRAVEL_PATH"      : 43,
  "WOODEN_TOWER"     : 44,
  "FIRE_STATION"     : 45,
  "CORN_FIELD"       : 46,
  "WINDMILL"         : 47,
  "BAKERY"           : 48,
  "SPICE_FIELD"      : 49,
  "BLENDER"          : 50,
  "BARRACKS"         : 53,
  "STONE_PIT"        : 54,
  "STONEMASON"       : 55,
  "COCOA_FIELD"      : 60,
  "VINEYARD"         : 61,
  "ALVEARIES"        : 62,
  "PASTRY_SHOP"      : 63,
  "WINERY"           : 65,
  "WEAPONSMITH"      : 66,
  "CANNON_FOUNDRY"   : 67,
  "BREWERY"          : 68,
  "HOP_FIELD"        : 69,
  "STONE_DEPOSIT"    : 70,
  "BARRIER"          : 71,
  "AMBIENT"          : 72,
  "SALINE"           : 86,
  "PUBLIC_BATH"      : 87,
}

# Reverse lookup dictionary: id -> name
id_to_name = {v: k for k, v in building_ids.items()}

# Source and destination directories
src_dir = Path(r"s:\src\unknown-horizons\content\gui\icons\buildmenu")
dst_dir = Path(r"s:\src\unknown-horizons\content\gui\icons\buildmenu_all")
dst_dir.mkdir(parents=True, exist_ok=True)

# Glob all PNG files that do NOT have "_h" in the stem
for file_path in src_dir.glob("*.png"):
    if "_h" in file_path.stem:
        continue  # skip hover versions

    # Extract the numeric prefix
    try:
        file_number = int(file_path.stem)
    except ValueError:
        print(f"Skipping {file_path.name}, does not match number pattern")
        continue

    # Reverse lookup
    building_name = id_to_name.get(file_number)
    if not building_name:
        print(f"No building found for {file_number}, skipping {file_path.name}")
        continue

    # Copy with new name
    new_file_name = f"{building_name.lower()}.png"
    shutil.copy(file_path, dst_dir / new_file_name)
    print(f"Copied {file_path.name} -> {new_file_name}")

In [ ]:
from pathlib import Path
from PIL import Image
import json
import math

# === Settings ===
src_folder = Path(r"s:\src\unknown-horizons\content\gui\icons\buildmenu_all")
grid_size = 47  # max tile size
output_image = src_folder / "buildmenu_all_icons.png"
output_json = src_folder / "buildmenu_all_icons.json"
output_snippets = src_folder / "buildmenu_all_icons.txt"
output_tab_snippets = src_folder / "buildmenu_all_icons_tabs.txt"
output_svg = src_folder / "buildmenu_all_icons.svg"

# Get all image files
image_files = list(sorted(src_folder.glob("*.png")))
image_files = [f for f in image_files if f != output_image]
# image_files = [f for f in image_files if f.suffix.lower() in ('.png', '.jpg', '.bmp')]

if not image_files:
  print("No image files found in folder.")
  exit()

num_images = len(image_files)
cols = int(math.ceil(math.sqrt(num_images)))
rows = int(math.ceil(num_images / cols))

atlas_width = cols * grid_size
atlas_height = rows * grid_size
atlas = Image.new("RGBA", (atlas_width, atlas_height), (0, 0, 0, 0))

metadata = []
snippets = []
tab_snippets = []
svg_paths = []

for idx, img_path in enumerate(image_files):
  img = Image.open(img_path).convert("RGBA")
  w, h = img.size

  if w > grid_size or h > grid_size:
    print(f"Warning: {img_path.name} is {w}x{h} and does not fit in {grid_size}x{grid_size}")

  col = idx % cols
  row = idx // cols
  x = col * grid_size
  y = row * grid_size

  atlas.paste(img, (x, y))

  # JSON metadata
  metadata.append({
    "file": img_path.name,
    "x": x,
    "y": y,
    "w": w,
    "h": h
  })

  # Text snippet
  file_name_safe = img_path.stem.replace(".", "_")
  snippet = f"""[sub_resource type="AtlasTexture" id="{file_name_safe}"]
atlas = ExtResource("buildmenu_all_icons_png")
region = Rect2({x}, {y}, {w}, {h})
"""
  snippets.append(snippet)
  tab_snippet = f"""[node name="Build_{file_name_safe.replace("bb_", "")}" parent="LeftFloatingPanel/TabSwitches" instance=ExtResource("3_switch_tab_widget")]
layout_mode = 2
texture_normal = SubResource("{file_name_safe}")
"""
  tab_snippets.append(tab_snippet)

  # SVG rectangle path
  svg_paths.append(f'  <rect id="{file_name_safe}" x="{x}" y="{y}" width="{w}" height="{h}" fill="none" stroke="red" />')

# Save atlas
atlas.save(output_image)
print(f"Atlas saved: {output_image}")

# Save JSON
with output_json.open("w") as f:
  json.dump(metadata, f, indent=2)
print(f"JSON metadata saved: {output_json}")

# Save text snippets
output_snippets.write_text("\n".join(snippets))
print(f"Text snippets saved: {output_snippets}")

# Save text snippets
output_tab_snippets.write_text("\n".join(tab_snippets))
print(f"Tab text snippets saved: {output_tab_snippets}")

# Save SVG
svg_content = f"""<svg xmlns="http://www.w3.org/2000/svg" width="{atlas_width}" height="{atlas_height}">
{chr(10).join(svg_paths)}
</svg>
"""
output_svg.write_text(svg_content)
print(f"SVG paths saved: {output_svg}")
